# PhysioGraph Final Clean

Minimal Colab control notebook. Core logic lives in `physiograph_colab_core.py`; this notebook mounts Drive, sets project paths, runs the workflow, and displays saved outputs.

## Provenance and scope (read first)

- **This notebook is the runnable deliverable** for PhysioGraph SpO2 drilldown review.
- **Results shown below are cached/precomputed** unless you run this notebook start-to-finish in Google Colab with `BUILD_NEW=True` and fresh data paths.
- **Stored notebook outputs do not claim fresh Colab execution.** Check `fresh_colab_execution` in `run_status.json` and `spo2_drilldown/manifest.json`.
- **SpO2 analyses are observational associations**, not causal effects.
- **Respiratory/RRT controls are text-derived proxies**, not structured ventilator or dialysis records.
- **Pulse-ox instability metrics are heuristic/proxy measures**, not validated clinical scores.
- **Pooled grouped CV metrics are internal diagnostics**, not external validation.
- **CLIF dataset is out of scope** for this notebook.


In [1]:
# Mount Google Drive when running in Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception as exc:
    IN_COLAB = False
    print(f'Google Drive mount skipped outside Colab: {exc}')


Google Drive mount skipped outside Colab: Error: credential propagation was unsuccessful


In [2]:
from pathlib import Path
import sys

DRIVE_ROOT = Path('/content/drive/MyDrive') if IN_COLAB else Path.cwd().parent
PROJECT_ROOT = DRIVE_ROOT / 'Projects' / 'PhysioGraph' if IN_COLAB else Path.cwd()
MIMIC_ROOT = DRIVE_ROOT / 'Data' / 'MIMIC' / 'Full'
EICU_ROOT = DRIVE_ROOT / 'Data' / 'eICU' / 'Full'
OUTPUT_ROOT = PROJECT_ROOT / 'physiograph_outputs'
BUILD_NEW = False

for candidate in (PROJECT_ROOT, PROJECT_ROOT / 'src'):
    candidate = candidate.resolve()
    if str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
print('BUILD_NEW =', BUILD_NEW)


PROJECT_ROOT = /content
OUTPUT_ROOT = /content/physiograph_outputs
BUILD_NEW = False


In [3]:
import importlib
import json
import physiograph_colab_core

physiograph_colab_core = importlib.reload(physiograph_colab_core)
run_outputs = physiograph_colab_core.run_physiograph_colab(
    project_root=PROJECT_ROOT,
    build_new=BUILD_NEW,
    mimic_root=MIMIC_ROOT,
    eicu_root=EICU_ROOT,
    output_root=OUTPUT_ROOT,
    run_comparator=True,
)

SPO2_DIR = OUTPUT_ROOT / 'spo2_drilldown'
RUN_STATUS_PATH = OUTPUT_ROOT / 'run_status.json'
MANIFEST_PATH = SPO2_DIR / 'manifest.json'

def _load_json(path):
    if not path.exists():
        return None
    with open(path) as fh:
        return json.load(fh)

run_status = _load_json(RUN_STATUS_PATH)
manifest = _load_json(MANIFEST_PATH)

print('=== Run status / provenance ===')
if run_status:
    for key in ('fresh_colab_execution', 'build_new', 'output_root', 'project_root'):
        if key in run_status:
            print(f'  {key}: {run_status[key]}')
else:
    print(f'  run_status.json not found: {RUN_STATUS_PATH}')

if manifest:
    for key in ('fresh_colab_execution', 'build_new', 'primary_endpoint', 'analysis_layers'):
        if key in manifest:
            print(f'  manifest.{key}: {manifest[key]}')
    if manifest.get('warnings'):
        print('  manifest.warnings:')
        for w in manifest['warnings']:
            print(f'    - {w}')
    if manifest.get('notes'):
        print('  manifest.notes:')
        for n in manifest['notes']:
            print(f'    - {n}')
else:
    print(f'  manifest.json not found: {MANIFEST_PATH}')

run_outputs


ModuleNotFoundError: No module named 'physiograph_colab_core'

In [ ]:
import json
import pandas as pd
from IPython.display import display

SPO2_DIR = OUTPUT_ROOT / 'spo2_drilldown'
RUN_STATUS_PATH = OUTPUT_ROOT / 'run_status.json'
MANIFEST_PATH = SPO2_DIR / 'manifest.json'


def _load_json(path):
    if not path.exists():
        return None
    with open(path) as fh:
        return json.load(fh)


def _show_csv(label, filename, *, note=None, head=25):
    print(f'\n=== {label} ===')
    if note:
        print(note)
    path = SPO2_DIR / filename
    if path.exists():
        display(pd.read_csv(path).head(head))
    else:
        print(f'Missing: {path}')


# 1) Provenance / manifest / run status
print('=== Provenance / manifest / run status ===')
run_status = _load_json(RUN_STATUS_PATH)
manifest = _load_json(MANIFEST_PATH)
if run_status:
    display(pd.DataFrame([{
        'fresh_colab_execution': run_status.get('fresh_colab_execution'),
        'build_new': run_status.get('build_new'),
        'output_root': run_status.get('output_root'),
    }]))
else:
    print(f'run_status.json not found: {RUN_STATUS_PATH}')
if manifest:
    display(pd.DataFrame([{
        'fresh_colab_execution': manifest.get('fresh_colab_execution'),
        'build_new': manifest.get('build_new'),
        'primary_endpoint': manifest.get('primary_endpoint'),
        'analysis_layers': ', '.join(manifest.get('analysis_layers', [])),
    }]))
    if manifest.get('warnings'):
        print('manifest warnings:')
        for w in manifest['warnings']:
            print(f'  - {w}')
    if manifest.get('notes'):
        print('manifest notes:')
        for n in manifest['notes']:
            print(f'  - {n}')
else:
    print(f'manifest.json not found: {MANIFEST_PATH}')

# 2) Availability audit
_show_csv('SpO2 availability audit', 'spo2_availability_audit.csv')

# 3) Primary CV / model metrics (internal grouped / out-of-fold if generated)
_show_csv(
    'SpO2 model metrics (primary endpoint)',
    'spo2_model_metrics.csv',
    note=(
        'WARNING: metrics here are internal grouped / out-of-fold diagnostics when generated; '
        'not external validation and not causal performance claims.'
    ),
)

# 4) Adjusted OR / p-values per feature
_show_csv('Adjusted OR / p-values per feature', 'spo2_or_pvalues_adjusted_per_feature.csv')

# 5) Respiratory context (optional)
_show_csv('Respiratory context (text-derived proxies)', 'spo2_respiratory_context.csv')

# 6) Lactate negative results
_show_csv('Lactate negative-control results', 'spo2_lactate_negative_results.csv')

# 7) External cross-dataset (optional)
print('\n=== External cross-dataset check ===')
ext_path = SPO2_DIR / 'spo2_external_cross_dataset.csv'
if ext_path.exists():
    display(pd.read_csv(ext_path).head(25))
else:
    print('spo2_external_cross_dataset.csv unavailable (not generated in this output tree).')

# 8) Missingness negative control (optional)
print('\n=== Missingness negative control ===')
miss_path = SPO2_DIR / 'spo2_missingness_negative_control.csv'
if miss_path.exists():
    display(pd.read_csv(miss_path).head(25))
else:
    print('spo2_missingness_negative_control.csv not run / not present in this output tree.')

# 9) Claims linter warnings
_show_csv('Claims linter warnings', 'claims_linter_warnings.csv')

# 10) Exploratory / context tables
_show_csv('Descriptive summary (exploratory)', 'spo2_descriptive_summary.csv')
_show_csv('Variability group summary (exploratory)', 'spo2_variability_group_summary.csv')


In [ ]:
from IPython.display import Image, display

print('=== Exploratory figures (descriptive; not validation) ===')
FIGURE_DIR = SPO2_DIR / 'figures'
if FIGURE_DIR.exists():
    for path in sorted(FIGURE_DIR.glob('*.png')):
        print(path.name)
        display(Image(filename=str(path)))
else:
    print(f'No figures directory: {FIGURE_DIR}')


# Buffer


In [ ]:
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#